# 13 — MicroEPI Spike Detection (Blackrock NS6 only)

**Goal.** Estimate, per tetrode, *how many candidate units* the recording contains and *how often / when* they fire. Diagnostic only — clustering happens later.

**Trial-timing logic.** Trial onsets, ends, and per-condition QC come from the same canonical pipeline used for the ERSP and HG plots in notebook 11: `lf_micromacro.extract_trials_with_qc(...)`. `cond_groups[cond] = (ons, offs, tends)` arrays are in **sample units of `fs`** (here `fs = 30000`).

**Pipeline.**
1. Load NS6 fragments → continuous 30 kHz stream. Unused 'X' channels dropped via regex; tetrode grouping ALSO excludes any prefix starting with X for defense-in-depth.
2. Adaptive 50-Hz harmonic notch on `signals_micro` only (photodiode untouched).
3. Build a minimal `d_all` and runtime preset, run `extract_trials_with_qc` for canonical trial timing. Plot QC of detected onsets/offsets on the photodiode trace.
4. Per-tetrode spike detection (full session): bandpass 300–6000 Hz → intra-tetrode CAR → MAD threshold → snippet extraction. Saturation amplitudes rejected; spikes within ±100 ms of fragment boundaries dropped.
5. Per-tetrode diagnostics: rate over time, snippet overlay, ISI histogram, peak-wire distribution, spikes-per-trial.
6. Stim-locked PSTH per condition using the canonical `cond_groups`.
7. Save `MicroEPI-G-0X_spikes.h5` cache.

**Out of scope.** Clustering / sorting (use Combinato or Wave_clus on the saved snippets, or a future notebook 14).

In [ ]:
# import sys; print(sys.version)
# try:
#     import lf_trials_qc as mmf
#     print("OK:", hasattr(mmf, "extract_trials_with_qc"))
# except Exception:
#     import traceback; traceback.print_exc()


In [ ]:
# Imports + per-patient config
import os, sys, re
sys.path.insert(0, "functions")

import numpy as np
import matplotlib.pyplot as plt
import h5py
import pandas as pd

import lf_micro_io      as bb          # NS6 session loader (readable; neo.io.BlackrockIO -> uV)
import lf_notch         as nf          # per-channel adaptive mains notch  (NEW)
import lf_sort          as ls          # configurable per-contact/tetrode detection  (NEW)
import lf_trials_qc     as mmf       # canonical trial timing (extract_trials_with_qc) (readable rewrite)
from config             import MICROEPI_MAT_PRESETS as MICROEPI_PRESETS

%load_ext autoreload
%autoreload 2

# -----------------------------------------------------------------------------
# Patient / session config
# -----------------------------------------------------------------------------
SERVER_ROOT = r'\\nasac-m2.unige.ch\m-HumanNeuronLab'   # <-- TODO: set to your actual server root

patient_id     = "MicroEPI-G-06"
preset_key     = "G-06"
base_filename  = "20260128-212210-2"
file_suffixes  = ["31", "32", "33", "34", "35", "36"]
photodiode_channel = "ainp2"

data_root        = os.path.join(SERVER_ROOT, "DATARAW", "MICROEPI", patient_id)
blackrock_dir    = os.path.join(data_root, "raw_blackrock","20260128-212210")
behavioral_tsv   = os.path.join(data_root, 'tasks','exp1_lora_2026_01_29_withmicro',"prep",'sub-6854_task-LanguageMapping_datetime-29-1-2026(17h33m57s)_language-FRE_events.tsv')  # <-- adjust if needed
out_cache        = os.path.join( "outputs","13_microSpikeSorting",f"{patient_id}", f"{patient_id}_spikes.h5")
qc_dir           = os.path.join( "outputs","13_microSpikeSorting",f"{patient_id}")

# QC / timing knobs — same defaults as notebook 11
BASELINE_W   = (-0.5, 0.0)
MIN_STIM_S   = 0.8 # 0.5 standard
MAX_POST_S   = 5.0 #10 standard
IQR_K        = 1.5
PD_THRESHOLD = 0.40
RESPONSE_COL = "response_type"

# Adaptive notch (signals_micro only)
DO_NOTCH       = True
NOTCH_BASE_HZ  = 50.0
NOTCH_FMAX_HZ  = 1000.0       # only notch the mains comb up to here
                              # (higher harmonics are tiny + handled by intra-tetrode CAR; raise if needed)
NOTCH_REPEATS  = 1
NOTCH_Z_THRESH = 3.0
NOTCH_DF_SAFETY = 2.0     # widen each notch to catch peak skirts (1.0 = exact width)
NOTCH_PASSES    = 3       # re-notch residual peaks up to N passes (auto-stops once clean)

# Drop pattern for unused channels (X, X1, X2, X_1, x3, etc.)
DROP_MICRO_PATTERN = r"^[Xx]\d+[mM]\d+$"
TETRODE_EXCLUDE_PREFIXES = ("X", "x")   # defense-in-depth at the grouping stage

# Spike detection knobs - TIGHTENED FOR NOISE REDUCTION
SPIKE_LOW_HZ      = 300.0
SPIKE_HIGH_HZ     = 6000.0
K_MAD             = 6.0           # was 4.5 - much higher threshold
SPIKE_POLARITY    = "neg"
REFRACTORY_MS     = 1.5           # was 1.0 - longer refractory
SNIPPET_PRE_MS    = 0.5
SNIPPET_POST_MS   = 1.5
MIN_AMPLITUDE_UV  = 100.0       # amplitude floor: keep spikes reaching <= -100 uV (None to disable)
MAX_AMPLITUDE_UV  = 300.0         # was 1000 - tighter saturation guard
FRAGMENT_GUARD_MS = 100.0

# Optional band-pass of the CLEANED micro signal (None = keep wideband). Removes
# sub-300 Hz mains harmonics + drift outright; shapes the saved .h5/.bin + PSD QC.
BANDPASS_HZ    = (300.0, 6000.0)
BANDPASS_ORDER = 4

# Detection unit: "wire" = per contact (each tetrode wire independently);
#                 "tetrode" = merged one-event-per-neuron across the 4 wires
DETECTION_UNIT    = "wire"


print(f"patient        : {patient_id}")
print(f"preset_key     : {preset_key}")
print(f"blackrock_dir  : {blackrock_dir}")
print(f"behavioral_tsv : {behavioral_tsv}")
print(f"out_cache      : {out_cache}")
print(f"notch          : enabled={DO_NOTCH}  base={NOTCH_BASE_HZ} Hz  up to {NOTCH_FMAX_HZ} Hz")
print(f"drop pattern   : {DROP_MICRO_PATTERN}")
print(f"polarity       : {SPIKE_POLARITY}   max_amp_uv: {MAX_AMPLITUDE_UV}   frag_guard_ms: {FRAGMENT_GUARD_MS}")

# Trial-locking knob — controls cell 5 (spikes/trial summary) AND cell 7 (per-condition PSTH)
# Options:
#   "stim_on"   -> stim onset (PD onset; default — what notebook 11 uses)
#   "stim_off"  -> stim offset (PD offset)
#   "trial_end" -> behavioral trial end (response/duration end from TSV)
LOCK_TO : str = "stim_off"


In [ ]:
# ---- Load NS6 session + adaptive notch + sanity checks ----
session = bb.read_blackrock_session(
    blackrock_dir, base_filename, file_suffixes,
    drop_micro_pattern=DROP_MICRO_PATTERN,
    verbose=True,
)

fs            = session["fs"]
signals_micro = session["signals_micro"]
signals_ainp  = session["signals_ainp"]
names_micro   = session["names_micro"]
names_ainp    = session["names_ainp"]
duration_s    = session["duration_s"]
fragment_boundaries_s = session["fragment_boundaries_s"]

# ---- Optional: restrict to a subset of tetrodes for testing ----
# Set to None (or []) to use all kept micro channels.
KEEP_PREFIXES = ("ADm",)         # AD/HAD tetrodes only  (trailing comma -> real tuple!)

if KEEP_PREFIXES:
    keep_mask = np.array(
        [any(n.startswith(p) for p in KEEP_PREFIXES) for n in names_micro]
    )
    signals_micro = signals_micro[:, keep_mask]
    names_micro   = [n for n, k in zip(names_micro, keep_mask) if k]
    print(f"[subset] keeping {keep_mask.sum()} channels "
          f"with prefixes {KEEP_PREFIXES}: {names_micro}")
    assert keep_mask.sum() > 0, "No channels matched KEEP_PREFIXES"

# Sanity check 1: no X channels survived the drop
_x_survivors = [n for n in names_micro if re.match(DROP_MICRO_PATTERN, n)]
assert not _x_survivors, f"X channels still present after drop: {_x_survivors}"

# Sanity check 2: photodiode channel exists in ainp
assert photodiode_channel in names_ainp, (
    f"Photodiode channel '{photodiode_channel}' not in ainp set: {names_ainp}")

print(f"\nfs={fs} Hz, total duration={duration_s/60:.1f} min")
print(f"micro channels (kept): {names_micro}")
print(f"ainp  channels       : {names_ainp}")
print(f"fragment boundaries (s): {np.round(fragment_boundaries_s, 2).tolist()}")

# ---- Adaptive 50-Hz harmonic notch on signals_micro (photodiode left untouched) ----
if DO_NOTCH:
    print(f"\n[notch] adaptive mains notch on {signals_micro.shape[1]} micro channels: ")
    signals_micro, notch_audit = nf.notch_mains_harmonics(
        signals_micro, fs,
        base=NOTCH_BASE_HZ,
        max_hz=min(NOTCH_FMAX_HZ, 0.5 * fs),
        repeats=NOTCH_REPEATS,
        peak_z_thresh=NOTCH_Z_THRESH,
        df_safety=NOTCH_DF_SAFETY, max_passes=NOTCH_PASSES,
        mode="per_channel",          # <-- per-channel peaks (was per-group median PSD)
        return_audit=True,
        verbose=True,
    )
    print(f"[notch] done. signals_micro shape: {signals_micro.shape}")
else:
    print("\n[notch] skipped (DO_NOTCH=False)")

# ---- Optional band-pass of the cleaned micro signal (spike band) ----
if BANDPASS_HZ:
    signals_micro = ls.bandpass_spike_band(signals_micro, fs,
                                           low=BANDPASS_HZ[0], high=BANDPASS_HZ[1],
                                           order=BANDPASS_ORDER)
    print(f"[bandpass] {BANDPASS_HZ[0]:.0f}-{BANDPASS_HZ[1]:.0f} Hz applied to signals_micro")


In [ ]:
# --- PSD QC: before vs. after notch ---
signals_micro_pre = session["signals_micro"]
from scipy.signal import welch

def psd(x, fs, sec=60):
    n = min(x.shape[0], int(sec*fs))
    return welch(x[:n], fs=fs, nperseg=int(2*fs), axis=0)

f, P_pre  = psd(signals_micro_pre, fs)
_, P_post = psd(signals_micro,     fs)

fig, ax = plt.subplots(figsize=(100,4))
ax.loglog(f, P_pre.mean(1),  label="pre")
ax.loglog(f, P_post.mean(1), label="post")
for h in np.arange(50, fs/2, 50): ax.axvline(h, c='r', lw=.3, alpha=.4)
ax.set(xlabel="Hz", ylabel=r"PSD [$\mu V^2$/Hz]"); ax.legend(); ax.grid(alpha=.3)
plt.show()

# Residual harmonic peak vs local floor (dB); want post ≈ 0
for h in np.arange(50, min(800, fs/2), 50):
    pk  = lambda P: P[(f>=h-1)&(f<=h+1)].mean()
    fl  = lambda P: P[((f>=h-10)&(f<=h-2))|((f>=h+2)&(f<=h+10))].mean()
    r = lambda P: 10*np.log10(pk(P.mean(1))/fl(P.mean(1)))
    print(f"{h:4.0f} Hz  pre {r(P_pre):+5.1f} dB   post {r(P_post):+5.1f} dB")


# --- Per-channel residual QC: worst/mean-channel residual at each harmonic AFTER notch ---
# (the per-channel notch should drive this toward 0 dB on every channel that had a peak)
import numpy as _np
_f0s, _r_post = nf.harmonic_residual_db(signals_micro, fs, base=NOTCH_BASE_HZ,
                                        max_hz=min(800.0, 0.5 * fs))
print("\nPer-channel post-notch residual (dB) at harmonics  [worst / mean channel]:")
for _k, _f0 in enumerate(_f0s):
    _col = _r_post[:, _k]
    if _np.any(_np.isfinite(_col)):
        print(f"  {_f0:6.0f} Hz   worst {_np.nanmax(_col):+6.1f}   mean {_np.nanmean(_col):+6.1f}")


In [ ]:
# ---- Trial timing — same canonical pipeline as ERSP/HG (notebook 11) ----
pd_signal = bb.extract_analog_channel(session, photodiode_channel)

d_all = dict(
    data_ecog  = np.zeros((0, signals_micro.shape[0]), dtype=np.float32),
    data_micro = np.zeros((0, signals_micro.shape[0]), dtype=np.float32),
    chans_ecog = np.array([], dtype=object),
    chans_micro= np.array(names_micro, dtype=object),
    photodiode = pd_signal.astype(np.float32),
    fs         = float(fs),
)

base_preset = MICROEPI_PRESETS[preset_key]
preset = dict(base_preset)
preset["trig"]       = photodiode_channel
preset["time_range"] = (50, 1444)
# preset["flip"] = False   # uncomment if PD polarity needs inverting on raw NS6
print(f"runtime preset: trig={preset['trig']}, time_range={preset['time_range']}, "
      f"flip={preset.get('flip', False)}, n_trials={len(preset['trial_ids'])}, "
      f"fake={preset.get('fake_trials', [])}")

tr = mmf.extract_trials_with_qc(
    d_all, preset, behavioral_tsv,
    pid=patient_id,
    baseline_w=BASELINE_W,
    min_stim_s=MIN_STIM_S,
    max_post_s=MAX_POST_S,
    iqr_k=IQR_K,
    pd_threshold=PD_THRESHOLD,
    response_col=RESPONSE_COL,
    save_qc_tsv=True,
    qc_dir=qc_dir,
)

cond_groups = tr["cond_groups"]
on_abs      = tr["on_abs"]
off_abs     = tr["off_abs"]
print(f"\ntotal valid trial onsets: {on_abs.size}")
for cn, (ons, offs, tends) in cond_groups.items():
    print(f"  {cn:>10s}: {ons.size} trials  (first onset = {ons[0]/fs:.2f} s)")

# ---- QC plot: photodiode trace + detected onsets/offsets, first 60 s ----
fig, ax = plt.subplots(figsize=(14, 2.5))
t = np.arange(len(pd_signal)) / fs
mask = t <= 60.0
ax.plot(t[mask], pd_signal[mask], color='k', lw=0.5)
on_s_60  = on_abs[on_abs  / fs <= 60.0] / fs
off_s_60 = off_abs[off_abs / fs <= 60.0] / fs
for s in on_s_60:
    ax.axvline(s, color='orange', lw=0.6, alpha=0.7)
for s in off_s_60:
    ax.axvline(s, color='lime',   lw=0.6, alpha=0.7, ls='--')
ax.set_xlabel("Time (s)"); ax.set_ylabel(f"{photodiode_channel}")
ax.set_title(f"{patient_id} — PD onsets (orange) / offsets (green dashed) — first 60 s "
             f"({on_s_60.size}/{on_abs.size} onsets shown)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:

spikes_per_tetrode = ls.sort_session(
    signals_micro, names_micro, fs,
    detection_unit=DETECTION_UNIT,
    exclude_prefixes=TETRODE_EXCLUDE_PREFIXES,
    spike_low_hz=SPIKE_LOW_HZ, spike_high_hz=SPIKE_HIGH_HZ,
    k_mad=K_MAD, polarity=SPIKE_POLARITY, refractory_ms=REFRACTORY_MS,
    snippet_pre_ms=SNIPPET_PRE_MS, snippet_post_ms=SNIPPET_POST_MS,
    min_amplitude_uv=MIN_AMPLITUDE_UV, max_amplitude_uv=MAX_AMPLITUDE_UV,
    fragment_boundaries_s=fragment_boundaries_s,
    fragment_guard_ms=FRAGMENT_GUARD_MS,
    verbose=True,
)

print(f"\n{'='*64}")
print(f"DETECTION_UNIT = {DETECTION_UNIT!r}  ->  {len(spikes_per_tetrode)} unit(s)")
print(f"{'='*64}")
for _lbl, _d in spikes_per_tetrode.items():
    print(f"  {_lbl:>18s}: {_d['n_final']:>6d} spikes  (raw {_d['n_raw']})")


In [ ]:
# ---- Per-tetrode diagnostics: how many neurons / how often do they fire ----
n_tet = len(spikes_per_tetrode)
fig, axes = plt.subplots(n_tet, 4, figsize=(16, 3.0 * n_tet),
                          gridspec_kw=dict(width_ratios=[2.5, 1.2, 1.5, 1.0]))
if n_tet == 1:
    axes = axes[None, :]

# Refractory metric uses 2 ms cutoff (the detector enforces 1 ms by construction,
# so any rate <1 ms is tautologically zero — 2 ms is the biologically meaningful test)
REFRACTORY_METRIC_MS = 2.0

for ti, (tet_lbl, d) in enumerate(spikes_per_tetrode.items()):
    st  = d['spike_times_s']
    pc  = d['peak_channel']
    n_s = st.size
    rate_overall = n_s / duration_s

    ls.plot_spike_rate_over_time(
        st, duration_s, bin_s=10.0,
        ax=axes[ti, 0],
        title=f"{tet_lbl}  total n={n_s}  mean={rate_overall:.1f} Hz",
    )
    ls.plot_snippet_overlay(
        d['snippets'], fs, peak_channel=pc, max_show=1000,
        ax=axes[ti, 1], title=f"{tet_lbl}  snippets",
    )
    if n_s >= 2:
        isi_ms = np.diff(np.sort(st)) * 1000.0
        bins = np.logspace(-1, 4, 80)
        axes[ti, 2].hist(isi_ms, bins=bins, color='steelblue', alpha=0.85)
        axes[ti, 2].axvline(1.0, color='red',    ls='--', lw=0.8, label='1 ms (detector)')
        axes[ti, 2].axvline(REFRACTORY_METRIC_MS, color='orange', ls='--', lw=0.8,
                            label=f'{REFRACTORY_METRIC_MS:.0f} ms (metric)')
        rv = float((isi_ms < REFRACTORY_METRIC_MS).sum()) / isi_ms.size * 100.0
        axes[ti, 2].set_xscale('log')
        axes[ti, 2].set_xlabel('ISI (ms)')
        axes[ti, 2].set_ylabel('count')
        axes[ti, 2].set_title(f"ISI  refractory<{REFRACTORY_METRIC_MS:.0f}ms: {rv:.1f}%")
        axes[ti, 2].legend(fontsize=7, loc='upper right')
    else:
        axes[ti, 2].set_title("ISI (not enough spikes)")

    wire_names = d['wire_names']
    counts = np.array([(pc == ci).sum() for ci in range(len(wire_names))])
    axes[ti, 3].bar(range(len(wire_names)), counts,
                    color=[plt.get_cmap('tab10')(ci % 10) for ci in range(len(wire_names))])
    axes[ti, 3].set_xticks(range(len(wire_names)))
    axes[ti, 3].set_xticklabels(wire_names, rotation=45, fontsize=8)
    axes[ti, 3].set_ylabel('# spikes')
    axes[ti, 3].set_title("peak-wire dist")

plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# Anchor selection — drives cell 5 spikes/trial AND cell 7 PSTH
# -----------------------------------------------------------------------------
def get_anchor_samples(cond_groups, lock_to):
    """Return dict[cond_name] -> 1d ndarray of anchor sample indices for the chosen lock."""
    out = {}
    for cn, (ons, offs, tends) in cond_groups.items():
        if lock_to == "stim_on":
            out[cn] = np.asarray(ons,   dtype=np.int64)
        elif lock_to == "stim_off":
            out[cn] = np.asarray(offs,  dtype=np.int64)
        elif lock_to == "trial_end":
            out[cn] = np.asarray(tends, dtype=np.int64)
        else:
            raise ValueError(f"LOCK_TO must be 'stim_on'/'stim_off'/'trial_end', got {lock_to!r}")
    return out

anchor_samples_per_cond = get_anchor_samples(cond_groups, LOCK_TO)
all_anchor_samp = (np.concatenate(list(anchor_samples_per_cond.values()))
                   if anchor_samples_per_cond else np.array([], dtype=np.int64))
print(f"\nAnchor: LOCK_TO={LOCK_TO!r}  total anchors across conds: {all_anchor_samp.size}")
for cn, a in anchor_samples_per_cond.items():
    if a.size:
        print(f"  {cn:>10s}: {a.size} anchors  (first {LOCK_TO} = {a[0]/fs:.2f}s)")
    else:
        print(f"  {cn:>10s}: 0 anchors")

# ---- Per-tetrode summary including spikes-per-trial in [TRIAL_WINDOW_S] around the chosen anchor ----
TRIAL_WINDOW_S = (0.0, 1.0)   # window around the anchor, in seconds
print(f"\nPer-tetrode summary  (window {TRIAL_WINDOW_S} s around {LOCK_TO!r}):")
print(f"{'tet':>6s}  {'n_spikes':>9s}  {'rate':>7s}  {'refr<2ms':>9s}  {'wires':>6s}  "
      f"{'N_tr':>5s}  {'spt_mean':>8s}  {'spt_std':>8s}  {'spt_min':>7s}  {'spt_med':>7s}  {'spt_max':>7s}")

for tet_lbl, d in spikes_per_tetrode.items():
    st = d['spike_times_s']; pc = d['peak_channel']
    n_s = st.size
    rate = n_s / duration_s
    if n_s >= 2:
        isi_ms = np.diff(np.sort(st)) * 1000.0
        rv = float((isi_ms < REFRACTORY_METRIC_MS).sum()) / isi_ms.size * 100.0
    else:
        rv = float('nan')
    n_active_wires = int((np.bincount(pc.astype(int), minlength=len(d['wire_names'])) > 10).sum())
    if all_anchor_samp.size:
        ev_s = all_anchor_samp.astype(np.float64) / fs
        per_trial = np.array([
            int(np.sum((st >= ev + TRIAL_WINDOW_S[0]) & (st < ev + TRIAL_WINDOW_S[1])))
            for ev in ev_s])
        n_tr   = per_trial.size
        spt_mean = float(per_trial.mean())
        spt_std  = float(per_trial.std())
        spt_min  = int(per_trial.min())
        spt_med  = float(np.median(per_trial))
        spt_max  = int(per_trial.max())
    else:
        n_tr = 0
        spt_mean = spt_std = spt_med = float('nan')
        spt_min = spt_max = 0
    print(f"{tet_lbl:>6s}  {n_s:>9d}  {rate:>5.1f}Hz  {rv:>8.1f}%  "
          f"{n_active_wires}/{len(d['wire_names']):>4d}  "
          f"{n_tr:>5d}  {spt_mean:>8.2f}  {spt_std:>8.2f}  {spt_min:>7d}  {spt_med:>7.1f}  {spt_max:>7d}")


In [ ]:
# ---- Dot raster of spike activity (all units / contacts) ----
fig, ax = plt.subplots(figsize=(16, max(2, 0.35 * len(spikes_per_tetrode))))
ls.plot_activity_raster(spikes_per_tetrode, duration_s=duration_s, ax=ax,
                        title=f"{patient_id} - spike raster ({DETECTION_UNIT})")
# overlay stim ON windows by condition (skipped gracefully if no trials)
try:
    _cmap = plt.get_cmap('tab10')
    for _ci, (_cn, (_ons, _offs, _tends)) in enumerate(cond_groups.items()):
        for _o, _off in zip(_ons, _offs):
            ax.axvspan(_o / fs, _off / fs, color=_cmap(_ci % 10), alpha=0.10, lw=0)
except Exception:
    pass
plt.tight_layout(); plt.show()


In [ ]:
# (intentionally left blank — was a scratch placeholder in earlier versions)


In [ ]:
# ---- Per-condition PSTH locked to LOCK_TO (set in cell 1) ----
PSTH_WINDOW_S = (-1, 3)   # seconds around the anchor
PSTH_BIN_S    = 0.1

cond_names = list(cond_groups.keys())
n_cond = len(cond_names)
n_tet  = len(spikes_per_tetrode)

fig, axes = plt.subplots(n_tet, n_cond,
                          figsize=(3.4 * n_cond, 2.5 * n_tet),
                          sharex=True)
if n_tet == 1:
    axes = axes[None, :]
if n_cond == 1:
    axes = axes[:, None]

cmap = plt.get_cmap('tab10')
for ti, (tet_lbl, d) in enumerate(spikes_per_tetrode.items()):
    for ci, cname in enumerate(cond_names):
        anchor_samp = anchor_samples_per_cond[cname]
        ev_s = anchor_samp.astype(np.float64) / fs
        ls.plot_psth(
            d['spike_times_s'], ev_s,
            window_s=PSTH_WINDOW_S, bin_s=PSTH_BIN_S,
            ax=axes[ti, ci],
            title=f"{tet_lbl}  {cname}  ({LOCK_TO})",
            color=cmap(ci % 10),
        )
plt.tight_layout(); plt.show()


In [ ]:
# ---- Save .h5 cache ----
os.makedirs(os.path.dirname(out_cache), exist_ok=True)

with h5py.File(out_cache, 'w') as f:
    g_sess = f.create_group('session')
    g_sess.create_dataset('signals_micro', data=signals_micro, compression='gzip', compression_opts=4)
    g_sess.create_dataset('signals_ainp',  data=signals_ainp,  compression='gzip', compression_opts=4)
    g_sess.create_dataset('fragment_boundaries_s', data=fragment_boundaries_s)
    g_sess.attrs['fs']            = fs
    g_sess.attrs['duration_s']    = duration_s
    g_sess.attrs['names_micro']   = np.array(names_micro,  dtype='S')
    g_sess.attrs['names_ainp']    = np.array(names_ainp,   dtype='S')
    g_sess.attrs['notch_applied'] = DO_NOTCH

    g_tr = f.create_group('trials')
    g_tr.create_dataset('on_abs',  data=tr['on_abs'])
    g_tr.create_dataset('off_abs', data=tr['off_abs'])
    g_cg = g_tr.create_group('cond_groups')
    for cname, (ons, offs, tends) in cond_groups.items():
        g_c = g_cg.create_group(cname)
        g_c.create_dataset('ons',   data=ons)
        g_c.create_dataset('offs',  data=offs)
        g_c.create_dataset('tends', data=tends)

    g_sp = f.create_group('spikes')
    for tet_lbl, d in spikes_per_tetrode.items():
        g_t = g_sp.create_group(tet_lbl)
        g_t.create_dataset('spike_times_s', data=d['spike_times_s'])
        g_t.create_dataset('snippets',      data=d['snippets'], compression='gzip', compression_opts=4)
        g_t.create_dataset('peak_channel',  data=d['peak_channel'])
        g_t.create_dataset('thresholds',    data=d['thresholds'])
        g_t.attrs['wire_indices'] = np.array(d['wire_indices'])
        g_t.attrs['wire_names']   = np.array(d['wire_names'], dtype='S')

    f.attrs['patient_id']    = patient_id
    f.attrs['preset_key']    = preset_key
    f.attrs['base_filename'] = base_filename
    f.attrs['file_suffixes'] = np.array(file_suffixes, dtype='S')
    f.attrs['polarity']      = SPIKE_POLARITY
    f.attrs['k_mad']         = K_MAD

print(f"Saved: {out_cache}")
print(f"Size : {os.path.getsize(out_cache)/1e9:.2f} GB")

In [ ]:
# ---- Export the whole cleaned micro recording as a flat int16 binary (FBMdata) ----
# The downstream "data in the end": the notched wideband micro signal as a flat
# int16 .bin (the standard input for spike sorters like Combinato / Kilosort).
# Layout: int16 little-endian, C-order (n_samples, n_channels). A JSON sidecar
# makes it self-describing (fs, channel order, shape, dtype, uV<->int16 gain).
# Written under outputs/ (git-ignored) so it is never pushed to GitHub.
#
# NOTE: exports the CURRENT signals_micro (which is the KEEP_PREFIXES subset if that
# is set). For the whole probe, set KEEP_PREFIXES = None in cell 2 and re-run.

import json, datetime as _dt, re as _re

def _fbm_date(base):
    m = _re.match(r"(\d{4})(\d{2})(\d{2})", str(base))
    if m:
        yyyy, mm, dd = m.groups()
        return f"{dd}-{mm}-{yyyy[2:]}"          # DD-MM-YY (no '/' -> valid filename)
    return _dt.date.today().strftime("%d-%m-%y")

FBM_DATE     = _fbm_date(base_filename)          # e.g. "28-01-26"
FBM_BIN_GAIN = 1.0                               # uV per int16 unit (1.0 = store rounded uV)

os.makedirs(qc_dir, exist_ok=True)
fbm_bin_path  = os.path.join(qc_dir, f"FBMdata_{FBM_DATE}.bin")
fbm_meta_path = os.path.join(qc_dir, f"FBMdata_{FBM_DATE}.json")

_scaled = np.asarray(signals_micro, dtype=np.float64) / FBM_BIN_GAIN
_n_clip = int(np.sum((_scaled < -32768) | (_scaled > 32767)))
_int16  = np.clip(np.round(_scaled), -32768, 32767).astype("<i2")   # (n_samples, n_channels)
_int16.tofile(fbm_bin_path)

_meta = dict(
    file=os.path.basename(fbm_bin_path), dtype="int16", byte_order="little", order="C",
    shape=list(_int16.shape), axes=["n_samples", "n_channels"],
    fs=float(fs), n_samples=int(_int16.shape[0]), n_channels=int(_int16.shape[1]),
    channel_names=list(names_micro), units="microvolts",
    gain_uV_per_unit=float(FBM_BIN_GAIN), notch_applied=bool(DO_NOTCH),
    source_base_filename=base_filename,
    fragment_boundaries_s=[float(x) for x in fragment_boundaries_s],
    n_samples_clipped=_n_clip,
)
with open(fbm_meta_path, "w") as _f:
    json.dump(_meta, _f, indent=2)

print(f"[FBM .bin] {fbm_bin_path}")
print(f"           {_int16.shape[0]} samples x {_int16.shape[1]} ch int16 "
      f"({os.path.getsize(fbm_bin_path)/1e9:.2f} GB), gain={FBM_BIN_GAIN} uV/unit, clipped={_n_clip}")
print(f"[FBM .bin] sidecar: {fbm_meta_path}")
print("           reload: np.fromfile(path, dtype='<i2').reshape(n_samples, n_channels)")


# IF PREPED DATA: 

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# ---- Load cached spike sorting data ----
patient_id = "MicroEPI-G-06"  # adjust as needed
out_cache = f"outputs/{patient_id}_spikes.h5"  # adjust path if different

with h5py.File(out_cache, 'r') as f:
    # Load session parameters
    fs = float(f['session'].attrs['fs'])
    duration_s = float(f['session'].attrs['duration_s'])
    names_micro = [s.decode() for s in f['session'].attrs['names_micro']]

    # Reconstruct spikes_per_tetrode from the h5 structure
    spikes_per_tetrode = {}

    for tet_lbl in f['spikes'].keys():
        g_tet = f['spikes'][tet_lbl]
        spikes_per_tetrode[tet_lbl] = {
            'spike_times_s': g_tet['spike_times_s'][:],
            'snippets': g_tet['snippets'][:].transpose(0, 2, 1),  # (N,time,wires) -> (N,wires,time)
            'peak_channel': g_tet['peak_channel'][:],
            'thresholds': g_tet['thresholds'][:],
            'wire_indices': g_tet.attrs['wire_indices'],
            'wire_names': [s.decode() for s in g_tet.attrs['wire_names']]
        }

    # Snippet timing parameters (reconstruct from snippet shape and fs)
    # Assuming standard 0.5ms pre, 1.5ms post (adjust if you used different values)
    SNIPPET_PRE_MS = 0.5
    SNIPPET_POST_MS = 1.5

print(f"Loaded data from {out_cache}")
print(f"Session: {fs} Hz, {duration_s/60:.1f} min")
print(f"Tetrodes: {list(spikes_per_tetrode.keys())}")
for tet_lbl, data in spikes_per_tetrode.items():
    n_spikes = data['snippets'].shape[0]
    print(f"  {tet_lbl}: {n_spikes} spikes")

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# ---- Load cached spike sorting data ----
patient_id = "MicroEPI-G-06"  # adjust as needed
out_cache = f"outputs/13_microSpikeSorting/{patient_id}/{patient_id}_spikes.h5"  # adjust path if different

with h5py.File(out_cache, 'r') as f:
    # Load session parameters
    fs = float(f['session'].attrs['fs'])
    duration_s = float(f['session'].attrs['duration_s'])
    names_micro = [s.decode() for s in f['session'].attrs['names_micro']]

    # Reconstruct spikes_per_tetrode from the h5 structure
    spikes_per_tetrode = {}

    for tet_lbl in f['spikes'].keys():
        g_tet = f['spikes'][tet_lbl]
        spikes_per_tetrode[tet_lbl] = {
            'spike_times_s': g_tet['spike_times_s'][:],
            'snippets': g_tet['snippets'][:].transpose(0, 2, 1),  # (N,time,wires) -> (N,wires,time)
            'peak_channel': g_tet['peak_channel'][:],
            'thresholds': g_tet['thresholds'][:],
            'wire_indices': g_tet.attrs['wire_indices'],
            'wire_names': [s.decode() for s in g_tet.attrs['wire_names']]
        }

    # Snippet timing parameters (reconstruct from snippet shape and fs)
    # Assuming standard 0.5ms pre, 1.5ms post (adjust if you used different values)
    SNIPPET_PRE_MS = 0.5
    SNIPPET_POST_MS = 1.5

print(f"Loaded data from {out_cache}")
print(f"Session: {fs} Hz, {duration_s/60:.1f} min")
print(f"Tetrodes: {list(spikes_per_tetrode.keys())}")
for tet_lbl, data in spikes_per_tetrode.items():
    n_spikes = data['snippets'].shape[0]
    print(f"  {tet_lbl}: {n_spikes} spikes")

In [ ]:
# ---- Enhanced Per-Tetrode Spike Sorting Diagnostics ----
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np

n_tetrodes = len(spikes_per_tetrode)
fig, axes = plt.subplots(n_tetrodes, 5, figsize=(20, 4*n_tetrodes))
if n_tetrodes == 1:
    axes = axes.reshape(1, -1)

for i, (tet_lbl, data) in enumerate(spikes_per_tetrode.items()):
    snippets = data['snippets']
    spike_times_s = data['spike_times_s']
    peak_channel = data['peak_channel']
    thresholds = data['thresholds']
    wire_names = data['wire_names']

    if snippets.shape[0] == 0:
        for j in range(5):
            axes[i, j].text(0.5, 0.5, 'No spikes', ha='center', va='center', transform=axes[i, j].transAxes)
            axes[i, j].set_title(f'{tet_lbl} - No Data')
        continue

    # 1. Spike rate over time
    ax1 = axes[i, 0]
    time_bins = np.arange(0, spike_times_s.max() + 60, 60)  # 1-minute bins
    counts, _ = np.histogram(spike_times_s, bins=time_bins)
    ax1.plot(time_bins[:-1]/60, counts, 'b-', alpha=0.7)
    ax1.set_xlabel('Time (min)')
    ax1.set_ylabel('Spikes/min')
    ax1.set_title(f'{tet_lbl}: Rate over time')
    ax1.grid(True, alpha=0.3)

    # 2. 2D Feature scatter (Peak amplitude vs spike width)
    ax2 = axes[i, 1]
    features = []
    for j in range(snippets.shape[0]):
        pk_wire = peak_channel[j]
        waveform = snippets[j, pk_wire, :]
        # Peak amplitude
        peak_amp = np.max(np.abs(waveform))
        # Spike width (trough to peak time)
        trough_idx = np.argmin(waveform)
        peak_idx = trough_idx + np.argmax(waveform[trough_idx:])
        spike_width = (peak_idx - trough_idx) * 1000 / fs  # ms
        features.append([peak_amp, spike_width])

    features = np.array(features)

    # Simple clustering for visualization
    if len(features) > 10:
        kmeans = KMeans(n_clusters=min(3, len(features)//50 + 1), random_state=42)
        clusters = kmeans.fit_predict(features)
        scatter = ax2.scatter(features[:, 0], features[:, 1], c=clusters, alpha=0.6, s=2)
        ax2.set_xlabel('Peak Amplitude (μV)')
        ax2.set_ylabel('Spike Width (ms)')
        ax2.set_title(f'{tet_lbl}: Feature Space ({len(np.unique(clusters))} clusters)')
    else:
        ax2.scatter(features[:, 0], features[:, 1], alpha=0.6, s=2)
        ax2.set_xlabel('Peak Amplitude (μV)')
        ax2.set_ylabel('Spike Width (ms)')
        ax2.set_title(f'{tet_lbl}: Feature Space')
    ax2.grid(True, alpha=0.3)

    # 3. Template quality metrics
    ax3 = axes[i, 2]
    # Mean template ± std across peak channel
    peak_wires = [peak_channel[j] for j in range(len(peak_channel))]
    most_common_wire = max(set(peak_wires), key=peak_wires.count)

    templates_on_peak = []
    for j in range(snippets.shape[0]):
        if peak_channel[j] == most_common_wire:
            templates_on_peak.append(snippets[j, most_common_wire, :])

    if templates_on_peak:
        templates_on_peak = np.array(templates_on_peak)
        mean_template = np.mean(templates_on_peak, axis=0)
        std_template = np.std(templates_on_peak, axis=0)

        time_ms = np.linspace(-(SNIPPET_PRE_MS), SNIPPET_POST_MS, len(mean_template))
        ax3.plot(time_ms, mean_template, 'r-', lw=2, label='Mean')
        ax3.fill_between(time_ms, mean_template - std_template, 
                        mean_template + std_template, alpha=0.3, color='red')

        # Template SNR
        baseline_std = np.std(mean_template[:int(SNIPPET_PRE_MS * fs / 1000)])
        peak_amp = np.max(np.abs(mean_template))
        template_snr = peak_amp / baseline_std if baseline_std > 0 else 0

        ax3.set_xlabel('Time (ms)')
        ax3.set_ylabel('Amplitude (μV)')
        ax3.set_title(f'{tet_lbl}: Template (SNR={template_snr:.1f})')
        ax3.grid(True, alpha=0.3)
        ax3.axvline(0, color='k', linestyle='--', alpha=0.5)
    else:
        ax3.text(0.5, 0.5, 'No templates', ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title(f'{tet_lbl}: No Template Data')

    # 4. Advanced ISI histogram with contamination estimate
    ax4 = axes[i, 3]
    if len(spike_times_s) > 1:
        isis = np.diff(spike_times_s) * 1000  # Convert to ms

        # Histogram
        bins = np.logspace(np.log10(0.1), np.log10(1000), 50)
        counts, bin_edges = np.histogram(isis, bins=bins)
        ax4.stairs(counts, bin_edges, fill=True, alpha=0.7)
        ax4.set_xscale('log')
        ax4.set_xlabel('ISI (ms)')
        ax4.set_ylabel('Count')

        # Contamination estimate (ISI < 2ms)
        contamination_pct = 100 * np.sum(isis < 2.0) / len(isis)
        refractory_violations = np.sum(isis < 1.0)

        # Mark refractory period
        ax4.axvline(1.0, color='r', linestyle='--', alpha=0.7, label='1ms refractory')
        ax4.axvline(2.0, color='orange', linestyle='--', alpha=0.7, label='2ms')

        ax4.set_title(f'{tet_lbl}: ISI (Contam: {contamination_pct:.1f}%, Refrac viol: {refractory_violations})')
        ax4.legend(fontsize=8)
        ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, 'Insufficient spikes', ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title(f'{tet_lbl}: ISI - Not enough spikes')

    # 5. Peak wire distribution and spatial consistency
    ax5 = axes[i, 4]
    wire_counts = np.bincount(peak_channel, minlength=4)
    wire_labels = [f'Wire {j}\n({wire_names[j]})' for j in range(4)]

    bars = ax5.bar(range(4), wire_counts, alpha=0.7)
    ax5.set_xticks(range(4))
    ax5.set_xticklabels(wire_labels, rotation=45, ha='right', fontsize=8)
    ax5.set_ylabel('# Spikes')

    # Spatial consistency: dominant wire percentage
    dominant_pct = 100 * np.max(wire_counts) / np.sum(wire_counts) if np.sum(wire_counts) > 0 else 0
    ax5.set_title(f'{tet_lbl}: Wire Distribution (Dom: {dominant_pct:.0f}%)')
    ax5.grid(True, alpha=0.3, axis='y')

    # Color bars by count
    for bar, count in zip(bars, wire_counts):
        if count == np.max(wire_counts) and count > 0:
            bar.set_color('red')
        elif count > 0:
            bar.set_color('orange')
        else:
            bar.set_color('lightgray')

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n{'='*60}")
print(f"SPIKE SORTING QUALITY SUMMARY")
print(f"{'='*60}")
for tet_lbl, data in spikes_per_tetrode.items():
    n_spikes = data['snippets'].shape[0]
    if n_spikes > 0:
        spike_times_s = data['spike_times_s']
        peak_channel = data['peak_channel']

        # Rate statistics
        duration_min = (spike_times_s.max() - spike_times_s.min()) / 60 if len(spike_times_s) > 1 else 0
        avg_rate = n_spikes / duration_min if duration_min > 0 else 0

        # ISI contamination
        if len(spike_times_s) > 1:
            isis = np.diff(spike_times_s) * 1000
            contamination = 100 * np.sum(isis < 2.0) / len(isis)
        else:
            contamination = 0

        # Spatial consistency
        wire_counts = np.bincount(peak_channel, minlength=4)
        spatial_consistency = 100 * np.max(wire_counts) / n_spikes

        print(f"{tet_lbl:>8}: {n_spikes:>5} spikes | {avg_rate:>5.1f} Hz | "
              f"{contamination:>4.1f}% contam | {spatial_consistency:>4.0f}% spatial consistency")
    else:
        print(f"{tet_lbl:>8}: {n_spikes:>5} spikes | No activity detected")
print(f"{'='*60}")

In [ ]:
# ---- Enhanced Diagnostics with Waveform Overlay + Noise Analysis ----
# Assumes `spikes_per_tetrode` is loaded with snippets shaped (N, wires, time)
# (the loader transposes from on-disk (N, time, wires) -> (N, wires, time)).

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np

n_tetrodes = len(spikes_per_tetrode)
fig, axes = plt.subplots(n_tetrodes, 6, figsize=(24, 4 * n_tetrodes))  # 6 columns
if n_tetrodes == 1:
    axes = axes.reshape(1, -1)

for i, (tet_lbl, data) in enumerate(spikes_per_tetrode.items()):
    snippets      = data['snippets']            # (N, wires, time)
    spike_times_s = np.asarray(data['spike_times_s'])
    peak_channel  = np.asarray(data['peak_channel'], dtype=int)
    thresholds    = data['thresholds']
    wire_names    = data['wire_names']

    if snippets.shape[0] == 0:
        for j in range(6):
            axes[i, j].text(0.5, 0.5, 'No spikes',
                             ha='center', va='center', transform=axes[i, j].transAxes)
            axes[i, j].set_title(f'{tet_lbl} - No Data')
        continue

    n_wires = snippets.shape[1]
    n_time  = snippets.shape[2]
    pre_samples = int(round(SNIPPET_PRE_MS * fs / 1000.0))

    # ---------- 1. Spike rate over time ----------
    ax1 = axes[i, 0]
    time_bins = np.arange(0, spike_times_s.max() + 60, 60)  # 1-minute bins
    counts, _ = np.histogram(spike_times_s, bins=time_bins)
    ax1.plot(time_bins[:-1] / 60, counts, 'b-', alpha=0.7)
    ax1.set_xlabel('Time (min)')
    ax1.set_ylabel('Spikes/min')
    ax1.set_title(f'{tet_lbl}: Rate over time')
    ax1.grid(True, alpha=0.3)

    # ---------- 2. Waveform overlay on dominant wire ----------
    ax2 = axes[i, 1]
    most_common_wire = int(np.bincount(peak_channel, minlength=n_wires).argmax())

    n_show = min(200, snippets.shape[0])
    show_indices = np.random.choice(snippets.shape[0], n_show, replace=False)
    time_ms = np.linspace(-SNIPPET_PRE_MS, SNIPPET_POST_MS, n_time)

    for idx in show_indices:
        if peak_channel[idx] == most_common_wire:
            ax2.plot(time_ms, snippets[idx, most_common_wire, :], 'b-', alpha=0.1, lw=0.5)

    templates_on_peak = snippets[peak_channel == most_common_wire, most_common_wire, :]
    if len(templates_on_peak) > 0:
        mean_template = np.mean(templates_on_peak, axis=0)
        ax2.plot(time_ms, mean_template, 'r-', lw=3, label='Mean')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Amplitude (μV)')
    ax2.set_title(f'{tet_lbl}: Waveforms (Wire {most_common_wire})')
    ax2.grid(True, alpha=0.3)
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax2.legend()

    # ---------- 3. Feature scatter with quality coloring ----------
    ax3 = axes[i, 2]
    features      = np.empty((snippets.shape[0], 2), dtype=np.float32)
    quality_scores = np.empty(snippets.shape[0], dtype=np.float32)

    for j in range(snippets.shape[0]):
        pk_wire    = peak_channel[j]
        waveform   = snippets[j, pk_wire, :]
        peak_amp   = float(np.max(np.abs(waveform)))
        trough_idx = int(np.argmin(waveform))
        peak_idx   = trough_idx + int(np.argmax(waveform[trough_idx:]))
        spike_width = (peak_idx - trough_idx) * 1000.0 / fs  # ms

        baseline_std = float(np.std(waveform[:pre_samples])) if pre_samples > 0 else 0.0
        snr = peak_amp / (baseline_std + 1e-6)

        trough_val = abs(waveform[trough_idx])
        peak_val   = abs(waveform[peak_idx])
        pt_ratio   = peak_val / (trough_val + 1e-6)

        features[j]       = [peak_amp, spike_width]
        quality_scores[j] = snr * pt_ratio

    scatter = ax3.scatter(features[:, 0], features[:, 1], c=quality_scores,
                           alpha=0.6, s=3, cmap='viridis')
    quality_thresh = float(np.percentile(quality_scores, 75))  # top 25%
    ax3.set_xlabel('Peak Amplitude (μV)')
    ax3.set_ylabel('Spike Width (ms)')
    ax3.set_title(f'{tet_lbl}: Quality Score (thresh={quality_thresh:.1f})')
    plt.colorbar(scatter, ax=ax3, label='Quality Score')
    ax3.grid(True, alpha=0.3)

    # ---------- 4. Template + std envelope ----------
    ax4 = axes[i, 3]
    if len(templates_on_peak) > 0:
        mean_template = np.mean(templates_on_peak, axis=0)
        std_template  = np.std(templates_on_peak, axis=0)
        ax4.plot(time_ms, mean_template, 'r-', lw=2, label='Mean')
        ax4.fill_between(time_ms, mean_template - std_template,
                          mean_template + std_template, alpha=0.3, color='red')

        baseline_std = float(np.std(mean_template[:pre_samples])) if pre_samples > 0 else 0.0
        peak_amp     = float(np.max(np.abs(mean_template)))
        template_snr = peak_amp / baseline_std if baseline_std > 0 else 0.0

        ax4.set_xlabel('Time (ms)')
        ax4.set_ylabel('Amplitude (μV)')
        ax4.set_title(f'{tet_lbl}: Template (SNR={template_snr:.1f})')
        ax4.grid(True, alpha=0.3)
        ax4.axvline(0, color='k', linestyle='--', alpha=0.5)
    else:
        ax4.text(0.5, 0.5, 'No templates', ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title(f'{tet_lbl}: No Template Data')

    # ---------- 5. ISI distribution + contamination ----------
    ax5 = axes[i, 4]
    if len(spike_times_s) > 1:
        isis = np.diff(np.sort(spike_times_s)) * 1000.0
        bins = np.logspace(np.log10(0.1), np.log10(1000), 50)
        counts_isi, bin_edges = np.histogram(isis, bins=bins)
        ax5.stairs(counts_isi, bin_edges, fill=True, alpha=0.7)
        ax5.set_xscale('log')
        ax5.set_xlabel('ISI (ms)')
        ax5.set_ylabel('Count')

        contamination_pct = 100.0 * np.sum(isis < 2.0) / len(isis)
        refractory_violations = int(np.sum(isis < 1.0))

        ax5.axvline(1.0, color='r', linestyle='--', alpha=0.7, label='1ms')
        ax5.axvline(2.0, color='orange', linestyle='--', alpha=0.7, label='2ms')

        title_color = ('red'    if contamination_pct > 10 else
                       'orange' if contamination_pct > 2  else 'green')
        ax5.set_title(f'{tet_lbl}: Contam {contamination_pct:.1f}%', color=title_color)
        ax5.legend(fontsize=8)
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, 'Insufficient spikes',
                 ha='center', va='center', transform=ax5.transAxes)
        ax5.set_title(f'{tet_lbl}: ISI - Not enough spikes')

    # ---------- 6. Wire distribution ----------
    ax6 = axes[i, 5]
    wire_counts = np.bincount(peak_channel, minlength=n_wires)
    wire_labels = [f'W{j}\n({wire_names[j] if j < len(wire_names) else "?"})'
                   for j in range(n_wires)]
    ax6.bar(range(n_wires), wire_counts, alpha=0.7)
    ax6.set_xticks(range(n_wires))
    ax6.set_xticklabels(wire_labels, rotation=45, ha='right', fontsize=8)
    ax6.set_ylabel('# Spikes')

    dominant_pct = 100.0 * np.max(wire_counts) / np.sum(wire_counts) if np.sum(wire_counts) > 0 else 0.0
    title_color = ('green'  if dominant_pct > 70 else
                   'orange' if dominant_pct > 50 else 'red')
    ax6.set_title(f'{tet_lbl}: Spatial {dominant_pct:.0f}%', color=title_color)
    ax6.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# ---- Noise Reduction Recommendations ----
print(f"\n{'=' * 80}")
print("NOISE REDUCTION RECOMMENDATIONS")
print(f"{'=' * 80}")

for tet_lbl, data in spikes_per_tetrode.items():
    n_spikes = data['snippets'].shape[0]
    if n_spikes == 0:
        continue

    spike_times_s = np.asarray(data['spike_times_s'])
    peak_channel  = np.asarray(data['peak_channel'], dtype=int)
    n_wires       = data['snippets'].shape[1]

    # Current contamination
    if len(spike_times_s) > 1:
        isis = np.diff(np.sort(spike_times_s)) * 1000.0
        contamination = 100.0 * np.sum(isis < 2.0) / len(isis)
    else:
        contamination = 0.0

    # Spatial consistency
    wire_counts = np.bincount(peak_channel, minlength=n_wires)
    spatial_consistency = 100.0 * np.max(wire_counts) / n_spikes

    # Recommendations
    recommendations = []
    if contamination > 10:
        recommendations.append("RAISE K_MAD to 6.0+")
    if contamination > 5:
        recommendations.append("Increase SNR_MIN to 8.0+")
    if spatial_consistency < 50:
        recommendations.append("Add spatial consistency filter")
    if n_spikes > 2000:
        recommendations.append("Random subsample to 1000-2000")

    if contamination > 10 or spatial_consistency < 50:
        status = "🔴 HIGH NOISE"
    elif contamination > 2:
        status = "🟡 MODERATE"
    else:
        status = "🟢 CLEAN"

    print(f"{tet_lbl:>8} {status}: {contamination:>4.1f}% contam, {spatial_consistency:>3.0f}% spatial")
    if recommendations:
        print(f"         → {', '.join(recommendations)}")

print(f"{'=' * 80}")
print("SUGGESTED PARAMETER CHANGES:")
print("- K_MAD = 6.0  (was 4.5 — higher threshold)")
print("- SNR_MIN = 8.0  (was 5.0 — stricter quality)")
print("- Add: MIN_SPATIAL_CONSISTENCY = 60%")
print("- MAX_SNIPPETS_PER_TETRODE = 2000  (was 5000)")
print("- Consider: amplitude bounds 50-300 μV instead of 30-400")


In [ ]:
# ---- Per-tetrode fine-grained spike rate over time, with stim overlay ----
# A standalone, more detailed version of the rate-over-time panel from the
# main diagnostics cell. Uses small bins (default 1 s) and overlays trial
# on/off intervals (color-coded by condition) plus a thin photodiode panel
# at the top so you can visually correlate rate with the stimulus envelope.
#
# Required in scope: spikes_per_tetrode, fs.
# Optional in scope (will gracefully fall back to the .h5 cache if missing):
#     cond_groups   (dict[name] -> (ons, offs, tends), all in samples)
#     on_abs, off_abs  (sample indices)
#     pd_signal     (raw photodiode trace)

import numpy as np
import matplotlib.pyplot as plt
import h5py

# ---- Tunable parameters ----
RATE_BIN_S       = 1.0        # rate histogram bin (seconds)
PD_DOWNSAMPLE_HZ = 200.0      # downsample photodiode to this rate just for plotting
SHOW_PD_PANEL    = True       # set False to skip the photodiode panel
LEGEND_FONTSIZE  = 8

# ---- Pull stim timing from scope, or from the cached .h5 if not loaded ----
_have_cond = 'cond_groups' in dir() and cond_groups is not None
_have_pd   = 'pd_signal' in dir() and pd_signal is not None

if not _have_cond:
    try:
        with h5py.File(out_cache, 'r') as f:
            if 'trials' in f and 'cond_groups' in f['trials']:
                _cg = {}
                for cname in f['trials']['cond_groups'].keys():
                    g_c = f['trials']['cond_groups'][cname]
                    _cg[cname] = (g_c['ons'][:], g_c['offs'][:], g_c['tends'][:])
                cond_groups = _cg
                on_abs  = f['trials']['on_abs'][:]
                off_abs = f['trials']['off_abs'][:]
                _have_cond = True
                print(f"[stim] loaded {len(cond_groups)} conditions from {out_cache}")
            else:
                print("[stim] no trial timing found in cache - skipping overlays")
    except Exception as e:
        print(f"[stim] could not load trial timing ({e}) - skipping overlays")

# ---- Determine session duration from spike data if needed ----
_max_time_s = 0.0
for d in spikes_per_tetrode.values():
    if d['spike_times_s'].size:
        _max_time_s = max(_max_time_s, float(np.max(d['spike_times_s'])))
if _have_cond and 'off_abs' in dir() and off_abs.size:
    _max_time_s = max(_max_time_s, float(np.max(off_abs)) / fs)
if 'duration_s' in dir() and duration_s is not None:
    _max_time_s = max(_max_time_s, float(duration_s))
duration_total_s = _max_time_s + 5.0  # small pad

# ---- Color map for conditions ----
if _have_cond:
    _cmap = plt.get_cmap('tab10')
    cond_colors = {cname: _cmap(i % 10) for i, cname in enumerate(cond_groups.keys())}

# ---- Figure layout: optional thin PD row on top, then one row per tetrode ----
n_tet = len(spikes_per_tetrode)
extra_rows = 1 if (SHOW_PD_PANEL and _have_pd) else 0
fig, axes = plt.subplots(
    n_tet + extra_rows, 1,
    figsize=(8*16, 1.2 * extra_rows + 2.2 * n_tet),
    sharex=True,
    gridspec_kw=dict(height_ratios=[0.6] * extra_rows + [1.0] * n_tet) if extra_rows else None,
)
if (n_tet + extra_rows) == 1:
    axes = [axes]
axes = list(np.atleast_1d(axes))

# ---- Photodiode panel (top) ----
if SHOW_PD_PANEL and _have_pd:
    ax_pd = axes[0]
    step = max(1, int(round(fs / PD_DOWNSAMPLE_HZ)))
    pd_ds = pd_signal[::step]
    t_pd  = np.arange(pd_ds.size) * (step / fs)
    ax_pd.plot(t_pd, pd_ds, color='k', lw=0.6)
    ax_pd.set_ylabel('PD (a.u.)', fontsize=9)
    ax_pd.set_title('Photodiode trace (downsampled)', fontsize=10)
    ax_pd.grid(True, alpha=0.3)

# ---- Per-tetrode rate plots ----
bin_edges = np.arange(0.0, duration_total_s + RATE_BIN_S, RATE_BIN_S)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

for k, (tet_lbl, data) in enumerate(spikes_per_tetrode.items()):
    ax = axes[k + extra_rows]
    st = np.asarray(data['spike_times_s'])

    if st.size == 0:
        ax.text(0.5, 0.5, 'No spikes', ha='center', va='center', transform=ax.transAxes)
        ax.set_ylabel(f'{tet_lbl}\nrate (Hz)')
        continue

    counts, _ = np.histogram(st, bins=bin_edges)
    rate_hz = counts / RATE_BIN_S
    ax.plot(bin_centers, rate_hz, color='steelblue', lw=0.8)
    ax.fill_between(bin_centers, 0, rate_hz, color='steelblue', alpha=0.25, step='mid')
    ax.set_ylabel(f'{tet_lbl}\nrate (Hz)', fontsize=9)
    ax.grid(True, alpha=0.3)

    # ---- Overlay stim ON intervals, color-coded by condition ----
    if _have_cond:
        for cname, (ons, offs, tends) in cond_groups.items():
            color = cond_colors[cname]
            for o, off in zip(ons, offs):
                ax.axvspan(o / fs, off / fs, color=color, alpha=0.18, lw=0)

# ---- Single legend for conditions, placed on the top-most plot with rates ----
if _have_cond:
    handles = [plt.Rectangle((0, 0), 1, 1, color=cond_colors[c], alpha=0.4) for c in cond_groups.keys()]
    labels  = list(cond_groups.keys())
    target_ax = axes[extra_rows]  # first tetrode panel
    target_ax.legend(handles, labels, loc='upper right',
                     fontsize=LEGEND_FONTSIZE, ncol=min(len(labels), 4),
                     framealpha=0.85, title='Stim condition')

# ---- X axis: time in seconds, shared across all panels ----
axes[-1].set_xlabel('Time (s)')
axes[-1].set_xlim(0, duration_total_s)

plt.suptitle(f'Spike rate vs. stimulus  |  bin = {RATE_BIN_S:g} s', y=1.0, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Restrict spikes to the per-trial in-task window (baseline_start -> trial_end) ----
# Builds the union of [on - |BASELINE_W[0]|, tend] intervals across all conditions,
# then filters every tetrode's spike data IN PLACE to keep only spikes inside one of
# those intervals. Run this AFTER the loader cell and BEFORE the strict QC cell.
#
# Set RESTRICT_TO_TRIAL_WINDOWS = False to skip filtering (useful for A/B comparison).
# Set CUSTOM_WINDOWS_S to a list of (t0_s, t1_s) tuples to override the per-trial logic
#   entirely (e.g. CUSTOM_WINDOWS_S = [(108, 384), (424, 778), (912, 1438)]).
#
# Required in scope: spikes_per_tetrode, fs.
# Optional: cond_groups, BASELINE_W. If missing, falls back to .h5 cache.

import numpy as np
import h5py

RESTRICT_TO_TRIAL_WINDOWS = True
CUSTOM_WINDOWS_S          = None    # e.g. [(108, 384), (424, 778), (912, 1438)]
PRE_PAD_S                 = None    # if None, uses |BASELINE_W[0]| (or 0.5 fallback)
POST_PAD_S                = 0.0     # extra seconds AFTER tend to keep

# ---- Resolve baseline pre-pad ----
if PRE_PAD_S is None:
    try:
        PRE_PAD_S = abs(float(BASELINE_W[0]))
    except Exception:
        PRE_PAD_S = 0.5
        print(f"[restrict] BASELINE_W not found in scope; using PRE_PAD_S = {PRE_PAD_S} s")

# ---- Build the list of (t0_s, t1_s) intervals ----
intervals = []   # all in seconds

if CUSTOM_WINDOWS_S is not None:
    intervals = [(float(a), float(b)) for (a, b) in CUSTOM_WINDOWS_S]
    print(f"[restrict] using CUSTOM_WINDOWS_S: {intervals}")
else:
    _have_cond = 'cond_groups' in dir() and cond_groups is not None
    if not _have_cond:
        # Try loading from the .h5 cache
        try:
            with h5py.File(out_cache, 'r') as f:
                if 'trials' in f and 'cond_groups' in f['trials']:
                    _cg = {}
                    for cname in f['trials']['cond_groups'].keys():
                        g_c = f['trials']['cond_groups'][cname]
                        _cg[cname] = (g_c['ons'][:], g_c['offs'][:], g_c['tends'][:])
                    cond_groups = _cg
                    _have_cond = True
                    print(f"[restrict] loaded {len(cond_groups)} conditions from cache")
        except Exception as e:
            print(f"[restrict] could not load cond_groups ({e})")

    if not _have_cond:
        raise RuntimeError("No cond_groups available - cannot restrict to trial windows. "
                           "Set CUSTOM_WINDOWS_S or run the trial-timing cell.")

    # Build per-trial intervals from (ons, offs, tends), all in samples
    for cname, (ons, offs, tends) in cond_groups.items():
        ons   = np.asarray(ons,   dtype=np.float64)
        tends = np.asarray(tends, dtype=np.float64)
        for o, te in zip(ons, tends):
            t0 = o  / fs - PRE_PAD_S
            t1 = te / fs + POST_PAD_S
            if t1 > t0:
                intervals.append((max(0.0, t0), t1))

# ---- Merge overlapping/adjacent intervals (sort by start, then sweep) ----
def _merge_intervals(ivs):
    if not ivs:
        return []
    ivs = sorted(ivs)
    out = [list(ivs[0])]
    for a, b in ivs[1:]:
        if a <= out[-1][1]:
            out[-1][1] = max(out[-1][1], b)
        else:
            out.append([a, b])
    return [(a, b) for a, b in out]

trial_mask_intervals = _merge_intervals(intervals)
total_in_window_s = sum(b - a for a, b in trial_mask_intervals)
print(f"[restrict] {len(trial_mask_intervals)} merged intervals, "
      f"total in-window time = {total_in_window_s:.1f} s")

def in_trial_window(spike_times_s, ivs=trial_mask_intervals):
    """Return boolean mask: True for spikes inside any (t0, t1) interval."""
    st = np.asarray(spike_times_s)
    if st.size == 0 or not ivs:
        return np.zeros(st.size, dtype=bool)
    starts = np.array([a for a, _ in ivs])
    ends   = np.array([b for _, b in ivs])
    # binary search: for each spike, find the rightmost interval whose start <= spike
    idx = np.searchsorted(starts, st, side='right') - 1
    keep = (idx >= 0) & (st <= ends[np.clip(idx, 0, len(ends) - 1)])
    return keep

# ---- Apply the mask in place to spikes_per_tetrode ----
if RESTRICT_TO_TRIAL_WINDOWS:
    print(f"\n{'=' * 80}")
    print(f"RESTRICTING SPIKES TO IN-TRIAL WINDOWS  (pre-pad {PRE_PAD_S:g} s, post-pad {POST_PAD_S:g} s)")
    print(f"{'=' * 80}")

    for tet_lbl, data in spikes_per_tetrode.items():
        st = np.asarray(data['spike_times_s'])
        n_before = st.size
        if n_before == 0:
            print(f"  [{tet_lbl}] empty - skipping")
            continue
        keep = in_trial_window(st)
        data['spike_times_s'] = st[keep]
        data['snippets']      = data['snippets'][keep]
        data['peak_channel']  = np.asarray(data['peak_channel'])[keep]
        n_after = data['spike_times_s'].size
        pct_kept = 100.0 * n_after / n_before if n_before else 0.0
        print(f"  [{tet_lbl}] {n_before} -> {n_after} spikes ({pct_kept:.1f}% kept)")
else:
    print("[restrict] RESTRICT_TO_TRIAL_WINDOWS = False - leaving spikes_per_tetrode untouched")
    print(f"           (intervals available via `trial_mask_intervals` and `in_trial_window`)")


In [ ]:
# ---- Apply ENHANCED quality filtering to cached snippets ----
# Operates on the already-loaded `spikes_per_tetrode` dict (from the .h5 cache).
# Filters: SNR, waveform shape, amplitude bounds, spatial consistency, subsample.

# pre-samples used when snippets were extracted (consistent with SNIPPET_PRE_MS, fs)
pre_samples_qc = int(round(SNIPPET_PRE_MS * fs / 1000.0))

# QC parameter set (strict)
SNR_MIN                 = .0
PEAK_TROUGH_RATIO_MIN   = 1.5
TROUGH_TO_PEAK_MS_MIN   = 0.25
TROUGH_TO_PEAK_MS_MAX   = 0.8
AMP_MIN_UV              = 40.0
AMP_MAX_UV              = 800.0
MIN_SPATIAL_CONSISTENCY = 60.0   # % of spikes on dominant wire
MAX_SNIPPETS_PER_TETRODE = 10000

np.random.seed(0)

print("=" * 80)
print("STRICT QUALITY FILTERING (per tetrode)")
print("=" * 80)

for tet_lbl, data in spikes_per_tetrode.items():
    snippets       = data['snippets']
    peak_ch_kept   = np.asarray(data['peak_channel'], dtype=int)
    spike_times_s  = np.asarray(data['spike_times_s'])

    n_before_qc = snippets.shape[0]
    if n_before_qc == 0:
        print(f"  [{tet_lbl}] no spikes — skipping")
        continue

    keep_quality = np.ones(n_before_qc, dtype=bool)

    # 1. SNR filter
    for i in range(n_before_qc):
        peak_wire    = peak_ch_kept[i]
        baseline_std = np.std(snippets[i, peak_wire, :pre_samples_qc])
        peak_amp     = np.abs(snippets[i, peak_wire, pre_samples_qc])
        snr = peak_amp / (baseline_std + 1e-6)
        if snr < SNR_MIN:
            keep_quality[i] = False

    # 2. Waveform shape filter
    for i in range(n_before_qc):
        peak_wire = peak_ch_kept[i]
        waveform  = snippets[i, peak_wire, :]
        trough_idx = int(np.argmin(waveform))
        peak_idx   = trough_idx + int(np.argmax(waveform[trough_idx:]))
        trough_val = abs(waveform[trough_idx])
        peak_val   = abs(waveform[peak_idx])
        if trough_val == 0 or peak_val / trough_val < PEAK_TROUGH_RATIO_MIN:
            keep_quality[i] = False
            continue
        trough_to_peak_ms = (peak_idx - trough_idx) * 1000.0 / fs
        if not (TROUGH_TO_PEAK_MS_MIN <= trough_to_peak_ms <= TROUGH_TO_PEAK_MS_MAX):
            keep_quality[i] = False

    # 3. Amplitude bounds
    for i in range(n_before_qc):
        peak_wire = peak_ch_kept[i]
        peak_amp  = np.max(np.abs(snippets[i, peak_wire, :]))
        if not (AMP_MIN_UV <= peak_amp <= AMP_MAX_UV):
            keep_quality[i] = False

    # 4. Spatial consistency filter
    n_wires = snippets.shape[1]
    wire_counts  = np.bincount(peak_ch_kept, minlength=n_wires)
    total_spikes = int(np.sum(wire_counts))
    spatial_consistency = (100.0 * np.max(wire_counts) / total_spikes) if total_spikes > 0 else 0.0
    if spatial_consistency < MIN_SPATIAL_CONSISTENCY and total_spikes > 20:
        dominant_wire = int(np.argmax(wire_counts))
        keep_quality &= (peak_ch_kept == dominant_wire)
        print(f"    [{tet_lbl}] spatial filter: enforcing dominant wire {dominant_wire} "
              f"(was {spatial_consistency:.1f}% consistent)")

    # 5. Random subsample
    n_after_filters = int(keep_quality.sum())
    if n_after_filters > MAX_SNIPPETS_PER_TETRODE:
        quality_indices  = np.flatnonzero(keep_quality)
        selected_indices = np.random.choice(quality_indices, MAX_SNIPPETS_PER_TETRODE, replace=False)
        keep_quality.fill(False)
        keep_quality[selected_indices] = True

    # Write filtered arrays back into the dict
    data['snippets']      = snippets[keep_quality]
    data['peak_channel']  = peak_ch_kept[keep_quality]
    data['spike_times_s'] = spike_times_s[keep_quality]

    n_final_qc    = int(keep_quality.sum())
    reduction_pct = 100.0 * (1.0 - n_final_qc / n_before_qc) if n_before_qc else 0.0
    print(f"  [{tet_lbl}] STRICT quality filter: {n_before_qc} -> {n_final_qc} snippets "
          f"({reduction_pct:.1f}% rejected | SNR>={SNR_MIN}, "
          f"spatial>={MIN_SPATIAL_CONSISTENCY}%, {AMP_MIN_UV}-{AMP_MAX_UV}uV)")

print("=" * 80)
print("Done. `spikes_per_tetrode` now contains only the surviving spikes.")


In [ ]:
# ---- Enhanced Diagnostics with Waveform Overlay + Noise Analysis ----
# Assumes `spikes_per_tetrode` is loaded with snippets shaped (N, wires, time)
# (the loader transposes from on-disk (N, time, wires) -> (N, wires, time)).

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np

n_tetrodes = len(spikes_per_tetrode)
fig, axes = plt.subplots(n_tetrodes, 6, figsize=(24, 4 * n_tetrodes))  # 6 columns
if n_tetrodes == 1:
    axes = axes.reshape(1, -1)

for i, (tet_lbl, data) in enumerate(spikes_per_tetrode.items()):
    snippets      = data['snippets']            # (N, wires, time)
    spike_times_s = np.asarray(data['spike_times_s'])
    peak_channel  = np.asarray(data['peak_channel'], dtype=int)
    thresholds    = data['thresholds']
    wire_names    = data['wire_names']

    if snippets.shape[0] == 0:
        for j in range(6):
            axes[i, j].text(0.5, 0.5, 'No spikes',
                             ha='center', va='center', transform=axes[i, j].transAxes)
            axes[i, j].set_title(f'{tet_lbl} - No Data')
        continue

    n_wires = snippets.shape[1]
    n_time  = snippets.shape[2]
    pre_samples = int(round(SNIPPET_PRE_MS * fs / 1000.0))

    # ---------- 1. Spike rate over time ----------
    ax1 = axes[i, 0]
    time_bins = np.arange(0, spike_times_s.max() + 60, 60)  # 1-minute bins
    counts, _ = np.histogram(spike_times_s, bins=time_bins)
    ax1.plot(time_bins[:-1] / 60, counts, 'b-', alpha=0.7)
    ax1.set_xlabel('Time (min)')
    ax1.set_ylabel('Spikes/min')
    ax1.set_title(f'{tet_lbl}: Rate over time')
    ax1.grid(True, alpha=0.3)

    # ---------- 2. Waveform overlay on dominant wire ----------
    ax2 = axes[i, 1]
    most_common_wire = int(np.bincount(peak_channel, minlength=n_wires).argmax())

    n_show = min(200, snippets.shape[0])
    show_indices = np.random.choice(snippets.shape[0], n_show, replace=False)
    time_ms = np.linspace(-SNIPPET_PRE_MS, SNIPPET_POST_MS, n_time)

    for idx in show_indices:
        if peak_channel[idx] == most_common_wire:
            ax2.plot(time_ms, snippets[idx, most_common_wire, :], 'b-', alpha=0.1, lw=0.5)

    templates_on_peak = snippets[peak_channel == most_common_wire, most_common_wire, :]
    if len(templates_on_peak) > 0:
        mean_template = np.mean(templates_on_peak, axis=0)
        ax2.plot(time_ms, mean_template, 'r-', lw=3, label='Mean')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Amplitude (μV)')
    ax2.set_title(f'{tet_lbl}: Waveforms (Wire {most_common_wire})')
    ax2.grid(True, alpha=0.3)
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax2.legend()

    # ---------- 3. Feature scatter with quality coloring ----------
    ax3 = axes[i, 2]
    features      = np.empty((snippets.shape[0], 2), dtype=np.float32)
    quality_scores = np.empty(snippets.shape[0], dtype=np.float32)

    for j in range(snippets.shape[0]):
        pk_wire    = peak_channel[j]
        waveform   = snippets[j, pk_wire, :]
        peak_amp   = float(np.max(np.abs(waveform)))
        trough_idx = int(np.argmin(waveform))
        peak_idx   = trough_idx + int(np.argmax(waveform[trough_idx:]))
        spike_width = (peak_idx - trough_idx) * 1000.0 / fs  # ms

        baseline_std = float(np.std(waveform[:pre_samples])) if pre_samples > 0 else 0.0
        snr = peak_amp / (baseline_std + 1e-6)

        trough_val = abs(waveform[trough_idx])
        peak_val   = abs(waveform[peak_idx])
        pt_ratio   = peak_val / (trough_val + 1e-6)

        features[j]       = [peak_amp, spike_width]
        quality_scores[j] = snr * pt_ratio

    scatter = ax3.scatter(features[:, 0], features[:, 1], c=quality_scores,
                           alpha=0.6, s=3, cmap='viridis')
    quality_thresh = float(np.percentile(quality_scores, 75))  # top 25%
    ax3.set_xlabel('Peak Amplitude (μV)')
    ax3.set_ylabel('Spike Width (ms)')
    ax3.set_title(f'{tet_lbl}: Quality Score (thresh={quality_thresh:.1f})')
    plt.colorbar(scatter, ax=ax3, label='Quality Score')
    ax3.grid(True, alpha=0.3)

    # ---------- 4. Template + std envelope ----------
    ax4 = axes[i, 3]
    if len(templates_on_peak) > 0:
        mean_template = np.mean(templates_on_peak, axis=0)
        std_template  = np.std(templates_on_peak, axis=0)
        ax4.plot(time_ms, mean_template, 'r-', lw=2, label='Mean')
        ax4.fill_between(time_ms, mean_template - std_template,
                          mean_template + std_template, alpha=0.3, color='red')

        baseline_std = float(np.std(mean_template[:pre_samples])) if pre_samples > 0 else 0.0
        peak_amp     = float(np.max(np.abs(mean_template)))
        template_snr = peak_amp / baseline_std if baseline_std > 0 else 0.0

        ax4.set_xlabel('Time (ms)')
        ax4.set_ylabel('Amplitude (μV)')
        ax4.set_title(f'{tet_lbl}: Template (SNR={template_snr:.1f})')
        ax4.grid(True, alpha=0.3)
        ax4.axvline(0, color='k', linestyle='--', alpha=0.5)
    else:
        ax4.text(0.5, 0.5, 'No templates', ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title(f'{tet_lbl}: No Template Data')

    # ---------- 5. ISI distribution + contamination ----------
    ax5 = axes[i, 4]
    if len(spike_times_s) > 1:
        isis = np.diff(np.sort(spike_times_s)) * 1000.0
        bins = np.logspace(np.log10(0.1), np.log10(1000), 50)
        counts_isi, bin_edges = np.histogram(isis, bins=bins)
        ax5.stairs(counts_isi, bin_edges, fill=True, alpha=0.7)
        ax5.set_xscale('log')
        ax5.set_xlabel('ISI (ms)')
        ax5.set_ylabel('Count')

        contamination_pct = 100.0 * np.sum(isis < 2.0) / len(isis)
        refractory_violations = int(np.sum(isis < 1.0))

        ax5.axvline(1.0, color='r', linestyle='--', alpha=0.7, label='1ms')
        ax5.axvline(2.0, color='orange', linestyle='--', alpha=0.7, label='2ms')

        title_color = ('red'    if contamination_pct > 10 else
                       'orange' if contamination_pct > 2  else 'green')
        ax5.set_title(f'{tet_lbl}: Contam {contamination_pct:.1f}%', color=title_color)
        ax5.legend(fontsize=8)
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, 'Insufficient spikes',
                 ha='center', va='center', transform=ax5.transAxes)
        ax5.set_title(f'{tet_lbl}: ISI - Not enough spikes')

    # ---------- 6. Wire distribution ----------
    ax6 = axes[i, 5]
    wire_counts = np.bincount(peak_channel, minlength=n_wires)
    wire_labels = [f'W{j}\n({wire_names[j] if j < len(wire_names) else "?"})'
                   for j in range(n_wires)]
    ax6.bar(range(n_wires), wire_counts, alpha=0.7)
    ax6.set_xticks(range(n_wires))
    ax6.set_xticklabels(wire_labels, rotation=45, ha='right', fontsize=8)
    ax6.set_ylabel('# Spikes')

    dominant_pct = 100.0 * np.max(wire_counts) / np.sum(wire_counts) if np.sum(wire_counts) > 0 else 0.0
    title_color = ('green'  if dominant_pct > 70 else
                   'orange' if dominant_pct > 50 else 'red')
    ax6.set_title(f'{tet_lbl}: Spatial {dominant_pct:.0f}%', color=title_color)
    ax6.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# ---- Noise Reduction Recommendations ----
print(f"\n{'=' * 80}")
print("NOISE REDUCTION RECOMMENDATIONS")
print(f"{'=' * 80}")

for tet_lbl, data in spikes_per_tetrode.items():
    n_spikes = data['snippets'].shape[0]
    if n_spikes == 0:
        continue

    spike_times_s = np.asarray(data['spike_times_s'])
    peak_channel  = np.asarray(data['peak_channel'], dtype=int)
    n_wires       = data['snippets'].shape[1]

    # Current contamination
    if len(spike_times_s) > 1:
        isis = np.diff(np.sort(spike_times_s)) * 1000.0
        contamination = 100.0 * np.sum(isis < 2.0) / len(isis)
    else:
        contamination = 0.0

    # Spatial consistency
    wire_counts = np.bincount(peak_channel, minlength=n_wires)
    spatial_consistency = 100.0 * np.max(wire_counts) / n_spikes

    # Recommendations
    recommendations = []
    if contamination > 10:
        recommendations.append("RAISE K_MAD to 6.0+")
    if contamination > 5:
        recommendations.append("Increase SNR_MIN to 8.0+")
    if spatial_consistency < 50:
        recommendations.append("Add spatial consistency filter")
    if n_spikes > 2000:
        recommendations.append("Random subsample to 1000-2000")

    if contamination > 10 or spatial_consistency < 50:
        status = "🔴 HIGH NOISE"
    elif contamination > 2:
        status = "🟡 MODERATE"
    else:
        status = "🟢 CLEAN"

    print(f"{tet_lbl:>8} {status}: {contamination:>4.1f}% contam, {spatial_consistency:>3.0f}% spatial")
    if recommendations:
        print(f"         → {', '.join(recommendations)}")

print(f"{'=' * 80}")
print("SUGGESTED PARAMETER CHANGES:")
print("- K_MAD = 6.0  (was 4.5 — higher threshold)")
print("- SNR_MIN = 8.0  (was 5.0 — stricter quality)")
print("- Add: MIN_SPATIAL_CONSISTENCY = 60%")
print("- MAX_SNIPPETS_PER_TETRODE = 2000  (was 5000)")
print("- Consider: amplitude bounds 50-300 μV instead of 30-400")


In [ ]:
# ---- Per-tetrode fine-grained spike rate over time, with stim overlay ----
# A standalone, more detailed version of the rate-over-time panel from the
# main diagnostics cell. Uses small bins (default 1 s) and overlays trial
# on/off intervals (color-coded by condition) plus a thin photodiode panel
# at the top so you can visually correlate rate with the stimulus envelope.
#
# Required in scope: spikes_per_tetrode, fs.
# Optional in scope (will gracefully fall back to the .h5 cache if missing):
#     cond_groups   (dict[name] -> (ons, offs, tends), all in samples)
#     on_abs, off_abs  (sample indices)
#     pd_signal     (raw photodiode trace)

import numpy as np
import matplotlib.pyplot as plt
import h5py

# ---- Tunable parameters ----
RATE_BIN_S       = 1.0        # rate histogram bin (seconds)
PD_DOWNSAMPLE_HZ = 200.0      # downsample photodiode to this rate just for plotting
SHOW_PD_PANEL    = True       # set False to skip the photodiode panel
LEGEND_FONTSIZE  = 8

# ---- Pull stim timing from scope, or from the cached .h5 if not loaded ----
_have_cond = 'cond_groups' in dir() and cond_groups is not None
_have_pd   = 'pd_signal' in dir() and pd_signal is not None

if not _have_cond:
    try:
        with h5py.File(out_cache, 'r') as f:
            if 'trials' in f and 'cond_groups' in f['trials']:
                _cg = {}
                for cname in f['trials']['cond_groups'].keys():
                    g_c = f['trials']['cond_groups'][cname]
                    _cg[cname] = (g_c['ons'][:], g_c['offs'][:], g_c['tends'][:])
                cond_groups = _cg
                on_abs  = f['trials']['on_abs'][:]
                off_abs = f['trials']['off_abs'][:]
                _have_cond = True
                print(f"[stim] loaded {len(cond_groups)} conditions from {out_cache}")
            else:
                print("[stim] no trial timing found in cache - skipping overlays")
    except Exception as e:
        print(f"[stim] could not load trial timing ({e}) - skipping overlays")

# ---- Determine session duration from spike data if needed ----
_max_time_s = 0.0
for d in spikes_per_tetrode.values():
    if d['spike_times_s'].size:
        _max_time_s = max(_max_time_s, float(np.max(d['spike_times_s'])))
if _have_cond and 'off_abs' in dir() and off_abs.size:
    _max_time_s = max(_max_time_s, float(np.max(off_abs)) / fs)
if 'duration_s' in dir() and duration_s is not None:
    _max_time_s = max(_max_time_s, float(duration_s))
duration_total_s = _max_time_s + 5.0  # small pad

# ---- Color map for conditions ----
if _have_cond:
    _cmap = plt.get_cmap('tab10')
    cond_colors = {cname: _cmap(i % 10) for i, cname in enumerate(cond_groups.keys())}

# ---- Figure layout: optional thin PD row on top, then one row per tetrode ----
n_tet = len(spikes_per_tetrode)
extra_rows = 1 if (SHOW_PD_PANEL and _have_pd) else 0
fig, axes = plt.subplots(
    n_tet + extra_rows, 1,
    figsize=(8*16, 1.2 * extra_rows + 2.2 * n_tet),
    sharex=True,
    gridspec_kw=dict(height_ratios=[0.6] * extra_rows + [1.0] * n_tet) if extra_rows else None,
)
if (n_tet + extra_rows) == 1:
    axes = [axes]
axes = list(np.atleast_1d(axes))

# ---- Photodiode panel (top) ----
if SHOW_PD_PANEL and _have_pd:
    ax_pd = axes[0]
    step = max(1, int(round(fs / PD_DOWNSAMPLE_HZ)))
    pd_ds = pd_signal[::step]
    t_pd  = np.arange(pd_ds.size) * (step / fs)
    ax_pd.plot(t_pd, pd_ds, color='k', lw=0.6)
    ax_pd.set_ylabel('PD (a.u.)', fontsize=9)
    ax_pd.set_title('Photodiode trace (downsampled)', fontsize=10)
    ax_pd.grid(True, alpha=0.3)

# ---- Per-tetrode rate plots ----
bin_edges = np.arange(0.0, duration_total_s + RATE_BIN_S, RATE_BIN_S)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

for k, (tet_lbl, data) in enumerate(spikes_per_tetrode.items()):
    ax = axes[k + extra_rows]
    st = np.asarray(data['spike_times_s'])

    if st.size == 0:
        ax.text(0.5, 0.5, 'No spikes', ha='center', va='center', transform=ax.transAxes)
        ax.set_ylabel(f'{tet_lbl}\nrate (Hz)')
        continue

    counts, _ = np.histogram(st, bins=bin_edges)
    rate_hz = counts / RATE_BIN_S
    ax.plot(bin_centers, rate_hz, color='steelblue', lw=0.8)
    ax.fill_between(bin_centers, 0, rate_hz, color='steelblue', alpha=0.25, step='mid')
    ax.set_ylabel(f'{tet_lbl}\nrate (Hz)', fontsize=9)
    ax.grid(True, alpha=0.3)

    # ---- Overlay stim ON intervals, color-coded by condition ----
    if _have_cond:
        for cname, (ons, offs, tends) in cond_groups.items():
            color = cond_colors[cname]
            for o, off in zip(ons, offs):
                ax.axvspan(o / fs, off / fs, color=color, alpha=0.18, lw=0)

# ---- Single legend for conditions, placed on the top-most plot with rates ----
if _have_cond:
    handles = [plt.Rectangle((0, 0), 1, 1, color=cond_colors[c], alpha=0.4) for c in cond_groups.keys()]
    labels  = list(cond_groups.keys())
    target_ax = axes[extra_rows]  # first tetrode panel
    target_ax.legend(handles, labels, loc='upper right',
                     fontsize=LEGEND_FONTSIZE, ncol=min(len(labels), 4),
                     framealpha=0.85, title='Stim condition')

# ---- X axis: time in seconds, shared across all panels ----
axes[-1].set_xlabel('Time (s)')
axes[-1].set_xlim(0, duration_total_s)

plt.suptitle(f'Spike rate vs. stimulus  |  bin = {RATE_BIN_S:g} s', y=1.0, fontsize=12)
plt.tight_layout()
plt.show()
